# 08. Điều tra lần chạy LSTM-E5 bất thường

Ở huấn luyện cuối (2019–2024, số epoch cố định = trung vị epoch tốt nhất trên CV), 1/5 hạt giống của LSTM-E5 cho NSE test
thấp bất thường. Notebook này kiểm tra: NaN/Inf trong dữ liệu và gradient, nổ gradient, chuẩn trọng số, hàm mất mát theo
epoch, trọng số mẫu, độ khớp trên chính tập train, khả năng tái lập, và **tần suất hỏng của quy trình số epoch cố định trên val
của CV** (không dùng test). Không loại hạt giống khỏi kết quả.

In [ ]:
# ── Google Colab: gắn Google Drive, vào thư mục dự án, cài đúng phiên bản thư viện. Chạy trên máy thì tự bỏ qua. ──
import os, sys, glob, subprocess
DU_AN = '/content/drive/MyDrive/T_DAThS_Ninh'   # ← thư mục dự án trên Google Drive (notebook 01, mục 0 tạo ra)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir(os.path.join(DU_AN, 'src')):          # không thấy → tự tìm trong My Drive (sâu tối đa 3 cấp)
        hit = [h for k in range(1, 4) for h in glob.glob('/content/drive/MyDrive/' + '*/' * k + 'src/results_store.py')]
        assert hit, ('Chưa có dự án trên Google Drive: mở notebook 01_PhanTich_XuLy_DuLieu và chạy 2 ô của mục 0 '
                     '(lấy code từ GitHub, giải nén data/kết quả), hoặc sửa biến DU_AN ở đầu ô này')
        DU_AN = os.path.dirname(os.path.dirname(hit[0]))
    os.chdir(DU_AN)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)

import json, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src' / 'results_store.py').exists():
    assert ROOT != ROOT.parent, 'Không tìm thấy thư mục dự án (thư mục chứa src/): mở notebook từ trong dự án'
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
from src.utils import load_config, results_dir, dataset_meta
cfg = load_config()
print('Thư mục dự án:', ROOT, '| kết quả lưu ở:', cfg['paths']['results'])
print('Thiết bị:', f'GPU {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else
      'CPU (trên Colab: Runtime → Change runtime type → GPU để chạy nhanh hơn)')
import torch
from src import pipeline
from src.data.feature_engineering import feature_list
from src.training.train import fit_predict
from src.training.losses import flow_weights
from src.evaluation.metrics import summary
OUT = results_dir(cfg, 'models', 'lstm', 'dieu_tra_E5')
from src import results_store as S
conf = pipeline.load_conf(cfg, 'lstm'); print(conf)
log = pd.read_csv(pipeline.out_dir(cfg, 'lstm') / 'train_log_H1.csv')
met = S.read_metrics(cfg, 'test', 1, 'chinh', 'lstm')
display(met[met.exp.isin(['E2', 'E5', 'E6'])][['exp', 'seed', 'n_iter', 'NSE', 'PBIAS', 'NSE_Q95', 'Peak_Bias']].round(3))

## 1. NaN / Inf, gradient, chuẩn trọng số, hàm mất mát theo epoch (huấn luyện cuối)

In [ ]:
print('Số batch có NaN/Inf (loss hoặc gradient) theo thực nghiệm:', log.groupby('exp').nonfinite_batches.sum().to_dict())
last = log.sort_values('epoch').groupby(['exp', 'seed']).tail(1).set_index(['exp', 'seed'])
display(last.loc[['E2', 'E5', 'E6'], ['epoch', 'train_loss', 'train_loss_max_batch', 'grad_norm_mean', 'grad_norm_max', 'weight_norm']].round(3))
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
for (e, s), d in log[log.exp == 'E5'].groupby(['exp', 'seed']):
    for ax, k in zip(axs, ['train_loss', 'grad_norm_max', 'weight_norm']):
        ax.plot(d.epoch, d[k], marker='o', ms=3, label=f'hạt giống {s}', lw=2.5 if s == met.loc[met.exp.eq('E5'), ['seed', 'NSE']].sort_values('NSE').seed.iloc[0] else 1)
for ax, k in zip(axs, ['hàm mất mát train (TB batch)', 'chuẩn gradient lớn nhất (trước khi cắt)', 'chuẩn trọng số']): ax.set_title(k); ax.set_xlabel('epoch')
axs[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(OUT / 'fig_log_huan_luyen_E5.png'); plt.show()

## 2. Dữ liệu và trọng số mẫu của cấu hình E5 (giai đoạn cuối)

In [ ]:
ds = pipeline.get_dataset(cfg, 1); feats = feature_list(cfg, 'F1'); hp = conf['hp']
v = ds.view('FINAL', feats, hp['L'])
print('X, Y hữu hạn:', np.isfinite(v.X_seq).all(), np.isfinite(v.Y).all(), '| số mẫu train:', len(v.idx_train))
w = flow_weights(v.W_q, v.M, v.idx_train, v.q95, conf['weights']['gamma'], conf['weights']['cap'])
wt = w[v.idx_train][v.M[v.idx_train]]
print(f"γ = {conf['weights']['gamma']}, Q95 train = {v.q95:.1f}: trọng số train min {wt.min():.2f}, trung vị {np.median(wt):.2f}, max {wt.max():.2f}; "
      f"{(wt > 5).sum()} ô có w > 5 (chiếm {100 * wt[wt > 5].sum() / wt.sum():.1f} % tổng trọng số)")

## 3. Tái lập và độ khớp trên chính tập train

Huấn luyện lại đúng cấu hình E5 cho từng hạt giống: (a) dự báo test phải trùng file đã lưu (tái lập); (b) NSE **trên tập train**
cho biết lần chạy hỏng là do tối ưu thất bại (khớp train kém) hay do quá khớp (khớp train tốt nhưng test kém).

In [ ]:
cvd = S.read_metrics(cfg, 'cv', 1, 'chinh', 'lstm'); n_e5 = max(1, int(round(cvd[cvd.exp == 'E5'].best_iter.median())))
saved = S.read_pred(cfg, 'test', 1, 'chinh', 'lstm')
rows = []
for s in cfg['seeds']['final']:
    P_te, _ = fit_predict(cfg, v, 'lstm', hp, s, conf['weights']['gamma'], conf['weights']['cap'], 0.0, n_iter=n_e5)
    P_tr, _ = fit_predict(cfg, v, 'lstm', hp, s, conf['weights']['gamma'], conf['weights']['cap'], 0.0, n_iter=n_e5, idx_eval=v.idx_train)
    old = saved[(saved.exp == 'E5') & (saved.seed == s)].sort_values('ngay_goc').Q_du_bao.values
    rows.append({'hạt giống': s, 'epoch': n_e5, 'tái lập (chênh lớn nhất, m³/s)': float(np.abs(P_te[:, 0] - old).max()),
                 'NSE train (trong mẫu)': summary(v.QO[v.idx_train], P_tr, v.M[v.idx_train])['NSE'],
                 'PBIAS train': summary(v.QO[v.idx_train], P_tr, v.M[v.idx_train])['PBIAS'],
                 'NSE test': summary(v.QO[v.idx_eval], P_te, v.M[v.idx_eval])['NSE']})
rep = pd.DataFrame(rows).set_index('hạt giống'); display(rep.round(3)); rep.to_csv(OUT / 'tai_lap_va_khop_train.csv', encoding='utf-8-sig')

## 4. Độ ổn định của quy trình số epoch cố định — trên val của CV (không dùng test)

Mỗi fold: huấn luyện với **đúng số epoch cố định** như giai đoạn cuối (không dừng sớm), 10 hạt giống, cho E2, E5, E6; chấm trên
val của fold. Nếu E5 có tỉ lệ "hỏng" cao hơn → bất ổn nằm ở quy trình (trọng số lớn + không dừng sớm), không phải lỗi dữ liệu.

In [ ]:
n_of = {e: max(1, int(round(cvd[cvd.exp == e].best_iter.median()))) for e in ['E2', 'E5', 'E6']}
spec = {'E2': (0, None, 0.0), 'E5': (conf['weights']['gamma'], conf['weights']['cap'], 0.0), 'E6': (0, None, conf['lambda'])}
rows = []
for f in ['F1', 'F2', 'F3']:
    vf = ds.view(f, feats, hp['L'])
    for e, (g_, c_, l_) in spec.items():
        for s in range(10):
            P, _ = fit_predict(cfg, vf, 'lstm', hp, s, g_, c_, l_, n_iter=n_of[e])
            rows.append({'fold': f, 'exp': e, 'seed': s, 'epoch': n_of[e], 'NSE val': summary(vf.QO[vf.idx_eval], P, vf.M[vf.idx_eval])['NSE']})
stab = pd.DataFrame(rows); stab.to_csv(OUT / 'on_dinh_epoch_co_dinh_cv.csv', index=False, encoding='utf-8-sig')
agg = stab.groupby(['fold', 'exp'])['NSE val'].agg(['mean', 'std', 'min', 'max'])
ref = stab.groupby(['fold', 'exp'])['NSE val'].transform('median')
stab['hỏng (< trung vị − 0,15)'] = stab['NSE val'] < ref - 0.15
agg['số lần hỏng /10'] = stab.groupby(['fold', 'exp'])['hỏng (< trung vị − 0,15)'].sum()
display(agg.round(3)); agg.to_csv(OUT / 'on_dinh_tom_tat.csv', encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(8, 3.6))
for i, (k, d) in enumerate(stab.groupby(['fold', 'exp'])):
    ax.scatter(np.full(len(d), i), d['NSE val'], s=12)
ax.set_xticks(range(len(agg))); ax.set_xticklabels([f'{a}\n{b}' for a, b in agg.index], fontsize=8); ax.set_ylabel('NSE val')
ax.set_title('LSTM, số epoch cố định, 10 hạt giống'); plt.tight_layout(); plt.savefig(OUT / 'fig_on_dinh_cv.png'); plt.show()

## 5. Lần chạy hỏng sai ở đâu trên test? (chẩn đoán, không dùng để chọn hay sửa)

Phân rã tổng bình phương sai số theo ngày; đánh dấu các ngày mà đầu vào Q CĐ(t) **vượt cực đại train** (ngoài miền huấn luyện).

In [ ]:
p = saved[saved.exp == 'E5']
W = p.pivot(index='ngay_goc', columns='seed', values='Q_du_bao'); o = p[p.seed == p.seed.min()].set_index('ngay_goc').Q_do
E2_ = W.sub(o, axis=0) ** 2
qmax_tr = float(np.nanmax(v.QO[v.idx_train])); qin = pd.Series(v.qlast[v.idx_eval], index=v.dates[v.idx_eval])
print(f'Cực đại Q đích train = {qmax_tr:.0f} m³/s; ngày gốc test có Q(t) > cực đại train:', list(qin[qin > qmax_tr].index.date))
print('Tổng bình phương sai số theo hạt giống:', E2_.sum().round(-4).to_dict())
bad = E2_.sum().idxmax()
top = E2_[bad].nlargest(6)
tab = pd.concat([o, qin.rename('Q(t) đầu vào'), W], axis=1).loc[top.index]
tab['tỉ trọng SSE (hạt giống hỏng)'] = (top / E2_[bad].sum()).values
display(tab.round(2)); tab.to_csv(OUT / 'sai_so_lon_nhat_hat_giong_hong.csv', encoding='utf-8-sig')
nxt = qin.index[qin > qmax_tr]
print(f'Tỉ trọng SSE của hạt giống {bad} ở 7 ngày sau các ngày đầu vào ngoài miền:',
      round(float(E2_[bad].loc[[d for d in E2_.index if any(0 <= (d - x).days <= 6 for x in nxt)]].sum() / E2_[bad].sum()), 3))